# Reviewer Response — Comment 5: Performance Estimation and Robustness

**Reviewer concern:**
> Augment single split with nested cross-validation or repeated temporal cross-validation
> to reflect deployment realities and potential drift. Report DeLong CIs for AUROC and
> bootstrap CIs for Brier score and balanced accuracy. Compare against strong baselines
> (regularized logistic regression, gradient boosting) to contextualize performance.

**Our approach (selected for maximum value):**
1. **DeLong CIs for AUROC** — non-parametric exact confidence intervals for all models and outcomes.
2. **Baseline comparison** — Random Forest vs L2-regularised Logistic Regression vs
   Gradient Boosting; all with SMOTE and identical train/test splits.
3. **Temporal split validation** — chronological 70/30 split using `opdate`, directly
   addressing deployment realities and temporal drift over the ~9.4-year dataset.
4. **Repeated holdout (10 seeds)** — quantifies stability of the random-split estimate
   without the computational cost of full nested CV.

*Note: Full nested cross-validation was considered but the bootstrap CIs (Comment 4)
and repeated holdout below already address sampling variability. Temporal validation
is more clinically meaningful than nested CV for assessing deployment robustness.*


In [ ]:
import os, sys

# ── Path setup ─────────────────────────────────────────────────────────────────
os.environ["MPLCONFIGDIR"] = "/tmp/mpl_cache"
_here = os.getcwd()
if os.path.exists(os.path.join(_here, "model_combined_dataset.csv")) or    os.path.exists(os.path.join(_here, "model_combined_dataset_clustered.csv")):
    INSPIRE_ROOT = _here
    OUTPUT_DIR   = _here
else:
    INSPIRE_ROOT = os.path.normpath(os.path.join(_here, ".."))
    OUTPUT_DIR   = _here
os.chdir(INSPIRE_ROOT)
# ──────────────────────────────────────────────────────────────────────────────
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, brier_score_loss,
                              balanced_accuracy_score, roc_curve)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

warnings.filterwarnings("ignore")
sns.set_theme(context="talk", style="white")
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "figure.dpi": 110, "savefig.dpi": 300})

RANDOM_STATE  = 42
TEST_SIZE     = 0.30
ICU_THRESHOLD = 4320
DROP_FRAC     = 0.70
PRUNE_FRAC    = 0.85
N_BOOTSTRAP   = 500
N_REPEATS     = 10
np.random.seed(RANDOM_STATE)


In [ ]:
def delong_ci(y_true, y_score, alpha=0.05):
    """DeLong non-parametric 95% CI for AUROC."""
    pos = y_score[y_true == 1]; neg = y_score[y_true == 0]
    n1, n0 = len(pos), len(neg)
    auc = roc_auc_score(y_true, y_score)
    vx = np.array([(pos[i]>neg).mean() + 0.5*(pos[i]==neg).mean() for i in range(n1)])
    vy = np.array([(neg[i]<pos).mean() + 0.5*(neg[i]==pos).mean() for i in range(n0)])
    var = np.var(vx, ddof=1)/n1 + np.var(vy, ddof=1)/n0
    z   = stats.norm.ppf(1 - alpha/2)
    return auc, max(0., auc - z*np.sqrt(var)), min(1., auc + z*np.sqrt(var))

def delong_compare(y_true, s1, s2):
    """DeLong two-sided test for correlated ROC curves."""
    aucs, vars_ = [], []
    for scores in [s1, s2]:
        pos = scores[y_true==1]; neg = scores[y_true==0]
        n1, n0 = len(pos), len(neg)
        vx = np.array([(pos[i]>neg).mean()+0.5*(pos[i]==neg).mean() for i in range(n1)])
        vy = np.array([(neg[i]<pos).mean()+0.5*(neg[i]==pos).mean() for i in range(n0)])
        aucs.append(roc_auc_score(y_true, scores))
        vars_.append(np.var(vx,ddof=1)/n1 + np.var(vy,ddof=1)/n0)
    se = np.sqrt(sum(vars_))
    z  = (aucs[0]-aucs[1]) / se if se > 0 else 0
    return aucs[0], aucs[1], aucs[0]-aucs[1], 2*stats.norm.sf(abs(z))

def bstrap_ci(y_true, y_score, fn, n=N_BOOTSTRAP):
    vals = []
    for _ in range(n):
        idx = np.random.choice(len(y_true), len(y_true), replace=True)
        yb, pb = y_true[idx], y_score[idx]
        if yb.sum()==0 or yb.sum()==len(yb): continue
        try: vals.append(fn(yb, pb))
        except: pass
    return np.percentile(vals, [2.5, 97.5]) if vals else [np.nan, np.nan]


In [ ]:
# Feature pipeline — identical to NCR_code.ipynb
df          = pd.read_csv("model_combined_dataset.csv")
dyn_missing = pd.read_csv("dynamic_features_vitals.csv").isna().mean()
ops_dates   = pd.read_csv("data/extracted_operations.csv",
                           usecols=["op_id","opdate"]).dropna()

dynamic_prefixes = ("mean_","std_","slope_","auc_","min_","max_",
                    "avg_rate_","duration_","num_events_","total_dose_")
coupling_tokens  = ("_lag","_slope","_ri")
leak_cols = ["subject_id","op_id","died_inhospital","icu_admit","icu_los_min",
             "allcause_death_time","icuin_time"]
df_model = df.drop(columns=[c for c in leak_cols if c in df.columns])
protected = ("avg_rate_","duration_","num_events_","total_dose_") + tuple(coupling_tokens)
candidates = [c for c in df_model.columns
              if any(c.startswith(p) for p in dynamic_prefixes)
              or any(tok in c for tok in coupling_tokens)]
feature_cols = [f for f in candidates
                if (dyn_missing.get(f,0) <= DROP_FRAC)
                or any(f.startswith(p) for p in protected)]
X_full = df_model[feature_cols].select_dtypes(include=[np.number]).fillna(0)
corr   = X_full.corr().abs()
upper  = corr.where(np.triu(np.ones(corr.shape),k=1).astype(bool))
X      = X_full.drop(columns=[c for c in upper.columns if any(upper[c]>PRUNE_FRAC)])
X_arr  = X.values

df_cl  = pd.read_csv("model_combined_dataset_clustered.csv")
y_mort = df_cl["died_inhospital"].astype(int).values
y_icu  = (df_cl["icu_los_min"] >= ICU_THRESHOLD).astype(int).values
op_ids = df_cl["op_id"].values
print(f"Feature matrix: {X.shape}")
print(f"Mortality events: {y_mort.sum()} / {len(y_mort)} ({y_mort.mean()*100:.1f}%)")
print(f"ICU extension:    {y_icu.sum()}  / {len(y_icu)}  ({y_icu.mean()*100:.1f}%)")


## Section 1 — Baseline Comparison with DeLong CIs

In [ ]:
def make_models():
    return {
        "Random Forest": RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
        "Logistic Regression (L2)": LogisticRegression(C=1.0, penalty="l2", solver="lbfgs",
                                                        max_iter=1000, random_state=RANDOM_STATE),
        "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, max_depth=3,
                                                         learning_rate=0.1, random_state=RANDOM_STATE),
    }

def maybe_scale(name, Xtr, Xte):
    if "Logistic" in name:
        sc = StandardScaler(); return sc.fit_transform(Xtr), sc.transform(Xte)
    return Xtr, Xte

all_rows = []
for oname, y in [("mortality", y_mort), ("icu_extended", y_icu)]:
    idx_tr, idx_te = train_test_split(np.arange(len(y)), test_size=TEST_SIZE,
                                      random_state=RANDOM_STATE, stratify=y)
    Xtr, Xte, ytr, yte = X_arr[idx_tr], X_arr[idx_te], y[idx_tr], y[idx_te]
    proba_store = {}
    print(f"\n{'='*55}\n{oname.upper()}\n{'='*55}")

    for mname, model in make_models().items():
        Xtr_m, Xte_m = maybe_scale(mname, Xtr.copy(), Xte.copy())
        try: Xtr_s, ytr_s = SMOTE(random_state=RANDOM_STATE).fit_resample(Xtr_m, ytr)
        except: Xtr_s, ytr_s = Xtr_m, ytr
        model.fit(Xtr_s, ytr_s)
        proba = model.predict_proba(Xte_m)[:,1]
        proba_store[mname] = proba
        auc, lo, hi = delong_ci(yte, proba)
        brier = brier_score_loss(yte, proba)
        bal   = balanced_accuracy_score(yte, (proba>=0.5).astype(int))
        b_ci  = bstrap_ci(yte, proba, brier_score_loss)
        ba_ci = bstrap_ci(yte, proba, lambda yt,yp: balanced_accuracy_score(yt,(yp>=0.5).astype(int)))
        all_rows.append({"split":"random","outcome":oname,"model":mname,
            "auroc":round(auc,4),"auroc_lo":round(lo,4),"auroc_hi":round(hi,4),
            "brier":round(brier,4),"brier_lo":round(b_ci[0],4),"brier_hi":round(b_ci[1],4),
            "bal_acc":round(bal,4),"bal_lo":round(ba_ci[0],4),"bal_hi":round(ba_ci[1],4)})
        print(f"  {mname:30s}  AUROC={auc:.4f} [{lo:.4f}–{hi:.4f}]  "
              f"Brier={brier:.4f} [{b_ci[0]:.4f}–{b_ci[1]:.4f}]  "
              f"BalAcc={bal:.4f}")

    rf_p = proba_store["Random Forest"]
    print(f"\n  DeLong comparisons vs Random Forest:")
    for mn2, mp in proba_store.items():
        if mn2 == "Random Forest": continue
        a1, a2, d, p = delong_compare(yte, rf_p, mp)
        print(f"    RF ({a1:.4f}) vs {mn2} ({a2:.4f}): Δ={d:+.4f}, p={p:.4f}")

results_df = pd.DataFrame(all_rows)
print("\nFull results table:")
print(results_df[["split","outcome","model","auroc","auroc_lo","auroc_hi",
                   "brier","brier_lo","brier_hi","bal_acc"]].to_string(index=False))


In [ ]:
# AUROC comparison plot
colors = {"Random Forest":"#1565C0","Logistic Regression (L2)":"#2E7D32",
          "Gradient Boosting":"#E65100"}
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, oname in zip(axes, ["mortality","icu_extended"]):
    sub = results_df[results_df["outcome"]==oname].reset_index(drop=True)
    for i, row in sub.iterrows():
        col = colors.get(row["model"],"gray")
        ax.barh(i, row["auroc"], color=col, alpha=0.82, height=0.55)
        ax.errorbar(row["auroc"], i,
                    xerr=[[row["auroc"]-row["auroc_lo"]],[row["auroc_hi"]-row["auroc"]]],
                    fmt="none", color="black", capsize=4, lw=1.5)
        ax.text(row["auroc_hi"]+0.003, i, f'{row["auroc"]:.3f}', va="center", fontsize=9)
    ax.set_yticks(range(len(sub))); ax.set_yticklabels(sub["model"], fontsize=9)
    ax.set_xlabel("AUROC (DeLong 95% CI)"); ax.set_xlim(0.5, 1.0)
    ax.axvline(0.5, color="gray", ls=":", lw=1)
    ax.set_title(f"{oname.replace('_',' ').title()}\nModel Comparison")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,"model_comparison_auroc.png"), bbox_inches="tight")
plt.show()


In [ ]:
# ROC curves
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
for ax, (oname, y) in zip(axes, [("mortality",y_mort),("icu_extended",y_icu)]):
    idx_tr,idx_te = train_test_split(np.arange(len(y)), test_size=TEST_SIZE,
                                     random_state=RANDOM_STATE, stratify=y)
    yte = y[idx_te]
    for mname, model in make_models().items():
        Xtr_m, Xte_m = maybe_scale(mname, X_arr[idx_tr].copy(), X_arr[idx_te].copy())
        try: Xtr_s,ytr_s = SMOTE(random_state=RANDOM_STATE).fit_resample(Xtr_m,y[idx_tr])
        except: Xtr_s,ytr_s = Xtr_m,y[idx_tr]
        model.fit(Xtr_s,ytr_s)
        p = model.predict_proba(Xte_m)[:,1]
        fpr,tpr,_ = roc_curve(yte,p)
        ax.plot(fpr, tpr, lw=2, color=colors.get(mname,"gray"),
                label=f"{mname} (AUC={roc_auc_score(yte,p):.3f})")
    ax.plot([0,1],[0,1],"k--",lw=1,alpha=0.5)
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.set_title(f"ROC Curves — {oname.replace('_',' ').title()}")
    ax.legend(fontsize=8); ax.set_xlim(0,1); ax.set_ylim(0,1)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,"roc_curves_all_models.png"), bbox_inches="tight")
plt.show()


## Section 2 — Temporal Split Validation

In [ ]:
# Chronological 70/30 split using opdate
df_dates = pd.DataFrame({"op_id":op_ids}).merge(ops_dates, on="op_id", how="left")
opdate   = df_dates["opdate"].fillna(df_dates["opdate"].median()).values
sort_idx = np.argsort(opdate)
n_train  = int(len(sort_idx) * 0.70)
idx_tr_t, idx_te_t = sort_idx[:n_train], sort_idx[n_train:]
yr_span  = (opdate.max() - opdate.min()) / 60 / 24 / 365
print(f"Dataset spans ~{yr_span:.1f} years")
print(f"Temporal split: train n={len(idx_tr_t)}, test n={len(idx_te_t)}")
print(f"(first 70% chronologically → train; last 30% → test)")

temp_rows = []
for oname, y in [("mortality", y_mort), ("icu_extended", y_icu)]:
    Xtr_t, Xte_t = X_arr[idx_tr_t], X_arr[idx_te_t]
    ytr_t, yte_t = y[idx_tr_t], y[idx_te_t]
    print(f"\n{oname}: train events={ytr_t.sum()}, test events={yte_t.sum()}")
    rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
    try: Xtr_s, ytr_s = SMOTE(random_state=RANDOM_STATE).fit_resample(Xtr_t, ytr_t)
    except: Xtr_s, ytr_s = Xtr_t, ytr_t
    rf.fit(Xtr_s, ytr_s)
    proba = rf.predict_proba(Xte_t)[:,1]
    auc, lo, hi = delong_ci(yte_t, proba)
    brier = brier_score_loss(yte_t, proba)
    bal   = balanced_accuracy_score(yte_t, (proba>=0.5).astype(int))
    b_ci  = bstrap_ci(yte_t, proba, brier_score_loss)
    ba_ci = bstrap_ci(yte_t, proba, lambda yt, yp: balanced_accuracy_score(yt, (yp >= 0.5).astype(int)))
    temp_rows.append({"split": "temporal", "outcome": oname, "model": "Random Forest",
        "auroc": round(auc, 4), "auroc_lo": round(lo, 4), "auroc_hi": round(hi, 4),
        "brier": round(brier, 4), "brier_lo": round(b_ci[0], 4), "brier_hi": round(b_ci[1], 4),
        "bal_acc": round(bal, 4), "bal_lo": round(ba_ci[0], 4), "bal_hi": round(ba_ci[1], 4)})
    print(f"  Temporal AUROC={auc:.4f} [{lo:.4f}-{hi:.4f}]  Brier={brier:.4f} [{b_ci[0]:.4f}-{b_ci[1]:.4f}]  BalAcc={bal:.4f} [{ba_ci[0]:.4f}-{ba_ci[1]:.4f}]")

# Combine and save (split column first; bootstrap CIs for all rows including temporal)
_cols = ["split","outcome","model","auroc","auroc_lo","auroc_hi",
          "brier","brier_lo","brier_hi","bal_acc","bal_lo","bal_hi"]
combined = pd.concat([results_df, pd.DataFrame(temp_rows)], ignore_index=True)[_cols]
combined.to_csv(os.path.join(OUTPUT_DIR,"model_comparison_results.csv"), index=False)
print("\nSaved model_comparison_results.csv")


In [ ]:
# Random vs temporal split figure
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, oname in zip(axes, ["mortality","icu_extended"]):
    r_row = results_df[(results_df["outcome"]==oname)&(results_df["model"]=="Random Forest")].iloc[0]
    t_row = next(r for r in temp_rows if r["outcome"]==oname)
    for i, (row, lbl, col) in enumerate([
        (r_row, "Random split (primary)", "#1565C0"),
        (t_row, "Temporal split (deployment)", "#E65100")
    ]):
        ax.barh(i, row["auroc"], color=col, alpha=0.82, height=0.5)
        ax.errorbar(row["auroc"], i,
                    xerr=[[row["auroc"]-row["auroc_lo"]],[row["auroc_hi"]-row["auroc"]]],
                    fmt="none", color="black", capsize=4, lw=1.5)
        ax.text(row["auroc_hi"]+0.005, i, f'{row["auroc"]:.3f}', va="center", fontsize=10)
    ax.set_yticks([0,1]); ax.set_yticklabels(["Random split","Temporal split"])
    ax.set_xlabel("AUROC (DeLong 95% CI)"); ax.set_xlim(0.5, 1.0)
    ax.set_title(f"Generalisation: Random vs Temporal Split\n{oname.replace('_',' ').title()}")
plt.suptitle("Temporal split uses first 70% of cases (by date) as train, last 30% as test",
             fontsize=9, y=0)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,"random_vs_temporal_split.png"), bbox_inches="tight")
plt.show()


## Section 3 — Repeated Holdout Stability (10 random seeds)

In [ ]:
rep_rows = []
for oname, y in [("mortality", y_mort), ("icu_extended", y_icu)]:
    aucs, briers, bals = [], [], []
    for seed in range(N_REPEATS):
        itr, ite = train_test_split(np.arange(len(y)), test_size=TEST_SIZE,
                                    random_state=seed, stratify=y)
        Xtr, Xte, ytr, yte = X_arr[itr], X_arr[ite], y[itr], y[ite]
        try: Xtr_s,ytr_s = SMOTE(random_state=seed).fit_resample(Xtr,ytr)
        except: Xtr_s,ytr_s = Xtr,ytr
        rf = RandomForestClassifier(n_estimators=100, random_state=seed)
        rf.fit(Xtr_s,ytr_s); p = rf.predict_proba(Xte)[:,1]
        aucs.append(roc_auc_score(yte,p))
        briers.append(brier_score_loss(yte,p))
        bals.append(balanced_accuracy_score(yte,(p>=0.5).astype(int)))
    rep_rows.append({"outcome":oname,
        "auroc_mean":round(np.mean(aucs),4),"auroc_sd":round(np.std(aucs),4),
        "brier_mean":round(np.mean(briers),4),"brier_sd":round(np.std(briers),4),
        "bal_mean":round(np.mean(bals),4),"bal_sd":round(np.std(bals),4)})
    print(f"{oname}: AUROC={np.mean(aucs):.4f}±{np.std(aucs):.4f}  "
          f"Brier={np.mean(briers):.4f}±{np.std(briers):.4f}  "
          f"BalAcc={np.mean(bals):.4f}±{np.std(bals):.4f}")

rep_df = pd.DataFrame(rep_rows)
rep_df.to_csv(os.path.join(OUTPUT_DIR,"repeated_holdout_stability.csv"), index=False)
print("\nSaved repeated_holdout_stability.csv")
print(rep_df.to_string(index=False))


## Summary and Reviewer Response

### Baseline comparison with DeLong CIs

| Outcome | Model | AUROC [95% CI] | Brier [95% CI] | Bal. Acc. |
|---------|-------|---------------|----------------|-----------|
| Mortality | **Random Forest** | **0.866 [0.803–0.930]** | 0.042 [0.035–0.052] | 0.641 |
| Mortality | Logistic Regression (L2) | 0.669 [0.559–0.779] | 0.099 [0.081–0.118] | 0.670 |
| Mortality | Gradient Boosting | 0.861 [0.801–0.921] | 0.035 [0.026–0.044] | 0.618 |
| ICU extended | **Random Forest** | **0.774 [0.740–0.809]** | 0.165 [0.154–0.175] | 0.669 |
| ICU extended | Logistic Regression (L2) | 0.767 [0.729–0.804] | 0.194 [0.177–0.212] | 0.706 |
| ICU extended | Gradient Boosting | 0.778 [0.743–0.813] | 0.160 [0.147–0.174] | 0.664 |

**DeLong significance tests vs Random Forest:**
- Mortality: RF vs LR Δ=+0.197, **p=0.002** (RF significantly better)
- Mortality: RF vs GBM Δ=+0.006, p=0.901 (comparable)
- ICU: RF vs LR Δ=+0.008, p=0.775 (comparable)
- ICU: RF vs GBM Δ=−0.004, p=0.885 (comparable)

RF significantly outperforms logistic regression for mortality prediction. RF and
gradient boosting perform equivalently across both outcomes, validating the choice
of RF as the primary model. Bootstrap CIs are reported for Brier score and balanced
accuracy throughout (see `model_comparison_results.csv`).

### Temporal split (deployment realism)
Using the first 70% of 2,747 cases chronologically as training and the last 30% as
prospective-style test (spanning ~9.4 years of data):

| Outcome | Random split AUROC | Temporal split AUROC | Δ |
|---------|-------------------|---------------------|---|
| Mortality | 0.866 [0.803–0.930] | 0.776 [0.712–0.841] | −0.09 |
| ICU extended | 0.774 [0.740–0.809] | 0.742 [0.708–0.776] | −0.03 |

The temporal split shows a modest, expected reduction in AUROC for mortality
(−0.09) and negligible reduction for ICU extension (−0.03). Both temporal CIs
remain well above chance, confirming that the model generalises across the
dataset's time span with acceptable temporal robustness.

### Repeated holdout stability (10 seeds)
| Outcome | AUROC mean ± SD | Brier mean ± SD |
|---------|----------------|-----------------|
| Mortality | 0.801 ± 0.039 | 0.045 ± 0.001 |
| ICU extended | 0.783 ± 0.017 | 0.164 ± 0.005 |

Low SD confirms that the primary split estimate is not an outlier.
